In [1]:
import os
import pandas as pd
import numpy as np

# Directory configuration
DATA_DIR = "./data"

# Test Configuration
EXPECTED_MODELS = [
    "meta-llama/Llama-3.2-1B", "meta-llama/Llama-3.2-3B",
    "EleutherAI/pythia-14m", "EleutherAI/pythia-31m", "EleutherAI/pythia-70m",
    "EleutherAI/pythia-160m", "EleutherAI/pythia-410m", "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b", "EleutherAI/pythia-2.8b",
    "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B", "Qwen/Qwen2.5-3B",
    "bigscience/bloom-560m", "bigscience/bloom-1b1", "bigscience/bloom-1b7", "bigscience/bloom-3b",
    "google/gemma-3-270m-it", "google/gemma-3-1b-it", "google/gemma-3-4b-it"
]
EXPECTED_SEEDS = [1, 2, 3, 4, 5]

# Dataset expectations
# 10 layers sampled via np.linspace
EXPECTED_LAYERS_COUNT = 10 
# "ind" is the target, leaving 6 languages as erased concepts
EXPECTED_CONCEPTS = ["eng", "dut", "jav", "sun", "ace", "ban"] 

# Row Count Calculations
# Erasure: (6 concepts + 1 baseline) * 10 layers = 70 rows
EXPECTED_ERASURE_ROWS = (len(EXPECTED_CONCEPTS) + 1) * EXPECTED_LAYERS_COUNT
# Validation: 6 concepts * 10 layers = 60 rows
EXPECTED_VALIDATION_ROWS = len(EXPECTED_CONCEPTS) * EXPECTED_LAYERS_COUNT

# Expected Schemas
ERASURE_SCHEMA = {
    'model': object, 'layer': np.number, 'erased_concept': object, 
    'nll_nats': np.number, 'n_tokens': np.number, 'n_bytes': np.number, 
    'ppl': np.number, 'bpb': np.number, 'bpb_delta': np.number, 
    'precision': object, 'seed': np.number
}

VALIDATION_SCHEMA = {
    'layer': np.number, 'erased_concept': object, 
    'probe_acc_before': np.number, 'probe_acc_after': np.number, 
    'probe_acc_drop': np.number, 'seed': np.number
}

print(f"Initialized validation parameters for {len(EXPECTED_MODELS)} models across {len(EXPECTED_SEEDS)} seeds.")

Initialized validation parameters for 20 models across 5 seeds.


In [2]:
import os
import pandas as pd
import numpy as np

# Directory configuration
DATA_DIR = "./data"

# Test Matrix Configuration
EXPECTED_MODELS = [
    "meta-llama/Llama-3.2-1B", "meta-llama/Llama-3.2-3B",
    "EleutherAI/pythia-14m", "EleutherAI/pythia-31m", "EleutherAI/pythia-70m",
    "EleutherAI/pythia-160m", "EleutherAI/pythia-410m", "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b", "EleutherAI/pythia-2.8b",
    "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B", "Qwen/Qwen2.5-3B",
    "bigscience/bloom-560m", "bigscience/bloom-1b1", "bigscience/bloom-1b7", "bigscience/bloom-3b",
    "google/gemma-3-270m-it", "google/gemma-3-1b-it", "google/gemma-3-4b-it"
]
EXPECTED_SEEDS = [1, 2, 3, 4, 5]

# Dataset expectations
EXPECTED_LAYERS_COUNT = 10 
# Corrected to include all 7 languages processed by the generation loop
EXPECTED_CONCEPTS = ["eng", "dut", "ind", "jav", "sun", "ace", "ban"] 

# Corrected Row Count Calculations
EXPECTED_ERASURE_ROWS = (len(EXPECTED_CONCEPTS) + 1) * EXPECTED_LAYERS_COUNT  # 80
EXPECTED_VALIDATION_ROWS = len(EXPECTED_CONCEPTS) * EXPECTED_LAYERS_COUNT     # 70

# Expected Schemas
ERASURE_SCHEMA = {
    'model': object, 'layer': np.number, 'erased_concept': object, 
    'nll_nats': np.number, 'n_tokens': np.number, 'n_bytes': np.number, 
    'ppl': np.number, 'bpb': np.number, 'bpb_delta': np.number, 
    'precision': object, 'seed': np.number
}

VALIDATION_SCHEMA = {
    'layer': np.number, 'erased_concept': object, 
    'probe_acc_before': np.number, 'probe_acc_after': np.number, 
    'probe_acc_drop': np.number, 'seed': np.number
}

print(f"Initialized validation parameters for {len(EXPECTED_MODELS)} models across {len(EXPECTED_SEEDS)} seeds.")
print(f"Expecting {EXPECTED_ERASURE_ROWS} erasure rows and {EXPECTED_VALIDATION_ROWS} validation rows per file.")

Initialized validation parameters for 20 models across 5 seeds.
Expecting 80 erasure rows and 70 validation rows per file.


In [3]:
missing_files = []
total_expected_files = len(EXPECTED_MODELS) * len(EXPECTED_SEEDS) * 2

for model in EXPECTED_MODELS:
    safe_name = model.replace("/", "_")
    for seed in EXPECTED_SEEDS:
        erasure_file = os.path.join(DATA_DIR, f"erasure_{safe_name}_seed{seed}.csv")
        validation_file = os.path.join(DATA_DIR, f"validation_{safe_name}_seed{seed}.csv")
        
        if not os.path.exists(erasure_file):
            missing_files.append(erasure_file)
        if not os.path.exists(validation_file):
            missing_files.append(validation_file)

if missing_files:
    print(f"CRITICAL: Found {len(missing_files)} missing files.")
    for f in missing_files[:10]:
        print(f"  - Missing: {f}")
    if len(missing_files) > 10:
        print("  ... and more.")
else:
    print(f"SUCCESS: All {total_expected_files} expected files are present.")

SUCCESS: All 200 expected files are present.


In [4]:
structural_errors = []

# Models that only contain 6 layers
SMALL_PYTHIA = ["EleutherAI/pythia-14m", "EleutherAI/pythia-31m", "EleutherAI/pythia-70m"]

for model in EXPECTED_MODELS:
    safe_name = model.replace("/", "_")
    
    # Adjust expected rows for 6-layer architectures
    if model in SMALL_PYTHIA:
        expected_e_rows = 6 * (len(EXPECTED_CONCEPTS) + 1) # 48
        expected_v_rows = 6 * len(EXPECTED_CONCEPTS)       # 42
    else:
        expected_e_rows = EXPECTED_ERASURE_ROWS            # 80
        expected_v_rows = EXPECTED_VALIDATION_ROWS         # 70

    for seed in EXPECTED_SEEDS:
        erasure_file = os.path.join(DATA_DIR, f"erasure_{safe_name}_seed{seed}.csv")
        validation_file = os.path.join(DATA_DIR, f"validation_{safe_name}_seed{seed}.csv")
        
        if not os.path.exists(erasure_file) or not os.path.exists(validation_file):
            continue
            
        try:
            df_e = pd.read_csv(erasure_file)
            df_v = pd.read_csv(validation_file)
            
            # Row Counts using the dynamically assigned expectations
            if len(df_e) != expected_e_rows:
                structural_errors.append(f"{erasure_file}: Expected {expected_e_rows} rows, got {len(df_e)}")
            if len(df_v) != expected_v_rows:
                structural_errors.append(f"{validation_file}: Expected {expected_v_rows} rows, got {len(df_v)}")
                
            # Column Signatures
            missing_e_cols = set(ERASURE_SCHEMA.keys()) - set(df_e.columns)
            if missing_e_cols:
                structural_errors.append(f"{erasure_file}: Missing columns {missing_e_cols}")
                
            missing_v_cols = set(VALIDATION_SCHEMA.keys()) - set(df_v.columns)
            if missing_v_cols:
                structural_errors.append(f"{validation_file}: Missing columns {missing_v_cols}")
                
        except Exception as e:
            structural_errors.append(f"Failed to read or parse files for {model} seed {seed}: {e}")

if structural_errors:
    print(f"CRITICAL: Found {len(structural_errors)} structural anomalies.")
    for err in structural_errors:
        print(f"  - {err}")
else:
    print("SUCCESS: All files passed structural and schema validation.")

SUCCESS: All files passed structural and schema validation.


In [5]:
sanity_errors = []

for model in EXPECTED_MODELS:
    safe_name = model.replace("/", "_")
    for seed in EXPECTED_SEEDS:
        erasure_file = os.path.join(DATA_DIR, f"erasure_{safe_name}_seed{seed}.csv")
        validation_file = os.path.join(DATA_DIR, f"validation_{safe_name}_seed{seed}.csv")
        
        if not os.path.exists(erasure_file) or not os.path.exists(validation_file):
            continue
            
        df_e = pd.read_csv(erasure_file)
        df_v = pd.read_csv(validation_file)
        
        # Erasure Sanity Checks
        if not df_e['ppl'].dropna().apply(lambda x: x >= 1.0 or np.isinf(x)).all():
            sanity_errors.append(f"{erasure_file}: Contains PPL values < 1.0")
            
        if not df_e['bpb'].dropna().apply(lambda x: x > 0 or np.isinf(x)).all():
            sanity_errors.append(f"{erasure_file}: Contains non-positive BPB values")
            
        if df_e.isnull().values.any():
            sanity_errors.append(f"{erasure_file}: Contains NaN values")
            
        # Validation Sanity Checks
        for col in ['probe_acc_before', 'probe_acc_after']:
            if not df_v[col].between(0, 1).all():
                sanity_errors.append(f"{validation_file}: {col} contains values outside [0, 1]")
                
        # Acc drop definition check (allowing for minor floating point imprecision)
        expected_drop = df_v['probe_acc_before'] - df_v['probe_acc_after']
        if not np.allclose(df_v['probe_acc_drop'], expected_drop, atol=1e-5):
            sanity_errors.append(f"{validation_file}: probe_acc_drop math mismatch")

if sanity_errors:
    print(f"CRITICAL: Found {len(sanity_errors)} logic/bound violations.")
    for err in sanity_errors[:10]:
        print(f"  - {err}")
else:
    print("SUCCESS: All data logic and bound sanity checks passed.")

SUCCESS: All data logic and bound sanity checks passed.
